# HW1: Decision Tree From Scratch (Entropy / Information Gain)

This notebook is for **loading data**, **training**, **evaluation**, and **visualization**.

You will implement core pieces of the decision tree in **`dt.py`** (see TODOs).  
Then run the cells below to verify your implementation.

---

## What you submit to Gradescope
- `dt.py`
- `hw1.ipynb`


## Google Colab Setup
1. Download the starter code files off of Canvas.
2. Create a folder for this HW1 programming assignment on your Google Drive and upload all of the starter code files into that folder.
3. One of the starter code files is a notebook .ipynb file. To open this file, create a blank Google Colab notebook, go to `File` > `Open Notebook`, and select the `hw1.ipynb` starter code file. You can now run and edit this notebook to complete the assignment!
4. In order to edit the `dt.py` file with colab, you will need to mount your Google Drive. To do this, run the following code cell. After running the cell, you will be prompted to authorize Google Colab to access your Google Drive. Follow the instructions to complete the authorization process.
5. Once your Google Drive is mounted, you can click the folder icon on the left sidebar to access your files. Navigate to the folder where you uploaded the starter code files, and you should see `dt.py` listed there. You can click on `dt.py` to open and edit it directly within Google Colab.

Note: When you make edits to `dt.py`, make sure to save the file and ***reimport*** the module in the notebook to see the changes reflected when you run the cells.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your folder
%cd /content/drive/MyDrive/<Path to your folder>
%ls 

In [ ]:
# (Colab) Install requirements
# If you are running locally and already installed dependencies, you can skip this cell.

import sys, subprocess, textwrap

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Minimal dependencies: numpy, pandas, matplotlib, scikit-learn
pip_install(["numpy", "pandas", "matplotlib", "scikit-learn"])
print("Done.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


In [ ]:
# Reload dt module after edits
import dt
import importlib
importlib.reload(dt)

## Load Dataset

Put `bank_auth.txt` in the current directory.


In [ ]:
def load_dataset(path, test_size=0.2, random_state=41):
    df = pd.read_csv(path, names=['variance','skewness','curtosis','entropy','label'])
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    return X_train, y_train, X_test, y_test

# Load dataset
test_size = 0.2
random_state = 41
X_train, y_train, X_test, y_test = load_dataset('bank_auth.txt', test_size=test_size, random_state=random_state)

print("Dataset loaded.")
print(f"Train samples: {X_train.shape}, Test samples: {X_test.shape}")
print(f"Train labels : {y_train.shape}, Test labels: {y_test.shape}")

## Train the Decision Tree

Hyperparameters:
- `max_depth`: maximum depth of the tree
- `min_samples_split`: minimum samples required to split a node
- `min_information_gain`: if best IG is below this, stop splitting


### How Continuous Features Are Handled in the Decision Tree (Optional Explanation)

The decision tree does **not** convert continuous features into binary features in advance. Instead, it creates a **binary split at each node using a threshold**. For a given feature $x_j$: The algorithm finds all unique values of that feature. It generates **candidate thresholds** as the midpoints between consecutive unique values: $\tau = \frac{v_k + v_{k+1}}{2}$. For each threshold $ \tau $, the feature is implicitly converted into a binary decision:
   - **Left child:** $ x_j \le \tau $
   - **Right child:** $ x_j > \tau $

Each threshold defines a binary split of the data. The algorithm evaluates all such splits and selects the one that produces the maximum information gain. This process is repeated recursively, and the same feature may be split using **different thresholds at different depths** of the tree.

In [ ]:
# Train
tree = dt.DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=2,
    min_information_gain=1e-9,
)

tree.fit(X_train, y_train)

train_acc = tree.score(X_train, y_train)
test_acc = tree.score(X_test, y_test)

print(f"Train accuracy: {train_acc*100:.1f}%")
print(f"Test accuracy:  {test_acc*100:.1f}%")


## Visualize the Learned Tree (Text)

This prints the tree structure up to a maximum print depth.


In [ ]:
feature_cols = ['variance','skewness','curtosis','entropy']
dt.print_tree(tree.root, feature_names=feature_cols, max_depth=6)

## Train a new tree on the same dataset with different train/test split

In [ ]:
test_size = 0.6
random_state = 40
X_train_new, y_train_new, X_test_new, y_test_new = load_dataset('bank_auth.txt', test_size=test_size, random_state=random_state)

# Train
tree_new = dt.DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=2,
    min_information_gain=1e-9,
)

tree_new.fit(X_train_new, y_train_new)

feature_cols = ['variance','skewness','curtosis','entropy']
dt.print_tree(tree_new.root, feature_names=feature_cols, max_depth=6)

## Effect of Tree Depth

In this question, you will explore how the **maximum depth** of a decision tree affects performance. What you need to do is to train decision trees with different values of `max_depth` and compare their accuracies.

In [ ]:
# Experiment for different tree depths

depths = [1, 2, 4, 8]

for d in depths:
    tree_d = dt.DecisionTreeClassifier(max_depth=d, min_samples_split=2, min_information_gain=1e-9)
    tree_d.fit(X_train, y_train)
    train_acc = tree_d.score(X_train, y_train) * 100
    test_acc = tree_d.score(X_test, y_test) * 100
    print(f"Max Depth: {d}, Train Accuracy: {train_acc:.1f}%, Test Accuracy: {test_acc:.1f}%")
